# Week 2: UCI Occupancy Detection - ARIMA Forecasting

Time series forecasting using real UCI occupancy sensor data with ARIMA model.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.arima.model import ARIMA
from datetime import timedelta
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load UCI occupancy dataset
occupancy_detection = fetch_ucirepo(id=357)
X = occupancy_detection.data.features
y = occupancy_detection.data.targets
df = pd.concat([X, y], axis=1)

# Clean data
mask = df['date'].astype(str).str.startswith('20')
df = df[mask].copy()
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Convert to numeric and drop NaN
for col in ['Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio', 'Occupancy']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.dropna(inplace=True)

print(f"Loaded {len(df)} rows")

Loaded 20560 rows


In [3]:
# Resample to hourly data
df_hourly = df.resample('H').agg({
    'Light': 'mean',
    'Occupancy': 'max'
}).dropna()
df_hourly.rename(columns={'Light': 'Power_Draw_Index'}, inplace=True)

# Split data for training
train_size = int(len(df_hourly) * 0.80)
train_data = df_hourly['Power_Draw_Index'][:train_size]
test_data = df_hourly['Power_Draw_Index'][train_size:]

In [4]:
# Train ARIMA model
model = ARIMA(train_data, order=(2, 1, 1))
model_fit = model.fit()

# Generate forecast
forecast_steps = len(test_data) + 24
forecast_result = model_fit.get_forecast(steps=forecast_steps)
forecast_values = forecast_result.predicted_mean
conf_int = forecast_result.conf_int()

# Clip negative predictions
forecast_values = forecast_values.apply(lambda x: max(x, 0))
conf_int[conf_int < 0] = 0

# Create time index for forecast
last_train_time = df_hourly.index[train_size - 1]
forecast_time_index = [last_train_time + timedelta(hours=x+1) for x in range(forecast_steps)]

print(f"Forecast generated for {forecast_steps} hours")

Forecast generated for 94 hours


In [5]:
# Visualization
fig = go.Figure()

# Historical data
fig.add_trace(go.Scatter(
    x=df_hourly.index[:train_size], 
    y=df_hourly['Power_Draw_Index'][:train_size],
    name="Historical Data",
    line=dict(color='gray', width=1)
))

# Actual test data
fig.add_trace(go.Scatter(
    x=df_hourly.index[train_size:], 
    y=df_hourly['Power_Draw_Index'][train_size:],
    name="Actual Observed",
    line=dict(color='orange', width=2)
))

# ARIMA forecast
fig.add_trace(go.Scatter(
    x=forecast_time_index,
    y=forecast_values,
    name="ARIMA Forecast",
    line=dict(color='blue', width=3)
))

# Confidence intervals
fig.add_trace(go.Scatter(
    x=forecast_time_index, 
    y=conf_int.iloc[:, 1],
    mode='lines',
    line=dict(width=0),
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=forecast_time_index, 
    y=conf_int.iloc[:, 0],
    mode='lines',
    line=dict(width=0),
    fill='tonexty',
    fillcolor='rgba(0, 0, 255, 0.2)',
    name="95% Confidence Interval"
))

fig.update_layout(
    title="UCI Occupancy Detection - ARIMA Forecast",
    yaxis_title="Light Intensity (Lux)",
    xaxis_title="Time",
    template="plotly_white",
    height=600
)

fig.show()